看看sar上涨下跌时候的平均涨幅或者跌幅

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
# 【关键两行】——必须同时设置！
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Heiti TC', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题[1][2][3][4][5][6][7][8][9][10]

import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/聚宽数据')
import datasource 
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.getAStockCodes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:4939,第一支股票:000001.XSHE


index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-17      0  6.65  6.65  6.59   6.63  3.263970e+08   
            2018-07-18      1  6.65  6.73  6.61   6.62  4.606974e+08   
            2018-07-19      2  6.65  6.70  6.61   6.64  4.036743e+08   
            2018-07-20      3  6.64  7.00  6.55   6.93  1.576844e+09   
            2018-07-23      4  6.87  7.19  6.84   7.16  1.646028e+09   

                             volume  
code        date                     
000001.XSHE 2018-07-17   49362461.0  
            2018-07-18   69061931.0  
            2018-07-19   60674284.0  
            2018-07-20  230701981.0  
            2018-07-23  234306736.0

In [4]:
help(talib.SAR)

Help on function SAR in module talib._ta_lib:

SAR(high, low, acceleration=0.02, maximum=0.2)
    SAR(high, low[, acceleration=?, maximum=?])

    Parabolic SAR (Overlap Studies)

    Inputs:
        prices: ['high', 'low']
    Parameters:
        acceleration: 0.02
        maximum: 0.2
    Outputs:
        real



In [5]:
dt['sar']=talib.SAR(dt['high'], dt['low']) # 这里是sar指标
# 未来1天的涨跌幅度
change_column_names = [] # 
for i in [1,2,3,4,5]:
    # 计算未来几天的涨幅是多少
    change_column_names.append(f'change_percent_{i}')
    dt[f'change_percent_{i}']=(dt['close'].shift(-i)-dt['close'])/dt['close']*100
dt.head()

index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-17      0  6.65  6.65  6.59   6.63  3.263970e+08   
            2018-07-18      1  6.65  6.73  6.61   6.62  4.606974e+08   
            2018-07-19      2  6.65  6.70  6.61   6.64  4.036743e+08   
            2018-07-20      3  6.64  7.00  6.55   6.93  1.576844e+09   
            2018-07-23      4  6.87  7.19  6.84   7.16  1.646028e+09   

                             volume     sar  change_percent_1  \
code        date                                                
000001.XSHE 2018-07-17   49362461.0     NaN         -0.150830   
            2018-07-18   69061931.0  6.5900          0.302115   
            2018-07-19   60674284.0  6.5928          4.367470   
            2018-07-20  230701981.0  7.0000          3.318903   
            2018-07-23  234306736.0  6.5500         -0.139665   

                        change_percent_2  change_percent_3  change_percent_4  \
code        date                                                               
000001.XSHE 2018-07-17          0.150830          4.524887          7.993967   
            2018-07-18          4.682779          8.157100          8.006042   
            2018-07-19          7.831325          7.680723          7.379518   
            2018-07-20          3.174603          2.886003          1.443001   
            2018-07-23         -0.418994         -1.815642         -1.815642   

                        change_percent_5  
code        date                          
000001.XSHE 2018-07-17          7.843137  
            2018-07-18          7.703927  
            2018-07-19          5.873494  
            2018-07-20          1.443001  
            2018-07-23         -0.279330

In [7]:
# 看看上涨或者下跌是的平均
# 先看上涨时候的
dt2 = dt.loc[dt['close'] > dt['sar'], :]
for i in [1,2,3,4,5]:
    _avg = dt2[f'change_percent_{i}'].mean()
    _median = dt2[f'change_percent_{i}'].median()
    print(f'sar上涨，股票均值:{_avg}，中位值:{_median}')

sar上涨，股票均值:0.04327679379841569，中位值:-0.09429533244772878
sar上涨，股票均值:0.0872547942068143，中位值:0.0
sar上涨，股票均值:0.1035232316904082，中位值:-0.11454753722794715
sar上涨，股票均值:0.09188932193002396，中位值:-0.09629328793593718
sar上涨，股票均值:0.07148402194594758，中位值:-0.2032520325203209


In [8]:
# 下跌
dt3 = dt.loc[dt['close'] < dt['sar'], :]
for i in [1,2,3,4,5]:
    _avg = dt3[f'change_percent_{i}'].mean()
    _median = dt3[f'change_percent_{i}'].median()
    print(f'sar下跌，股票均值:{_avg}，中位值:{_median}')

sar下跌，股票均值:0.045646180326108024，中位值:-0.07417178955650056
sar下跌，股票均值:0.08942807316053147，中位值:0.0
sar下跌，股票均值:0.15687844404906853，中位值:0.0
sar下跌，股票均值:0.2516197608041712，中位值:0.0
sar下跌，股票均值:0.35518176073818236，中位值:-0.07911392405063122


In [10]:
dt.tail(20)

index   open   high    low  close         money  \
code        date                                                          
000001.XSHE 2026-09-11   1980  11.57  11.60  11.46  11.49  9.799906e+08   
            2026-09-14   1981  11.48  11.64  11.47  11.59  8.760710e+08   
            2026-09-15   1982  11.57  11.62  11.52  11.57  8.933829e+08   
            2026-09-16   1983  11.55  11.58  11.32  11.45  1.106653e+09   
            2026-09-17   1984  11.43  11.49  11.32  11.36  8.061531e+08   
            2026-09-18   1985  11.34  11.57  11.31  11.45  9.999181e+08   
            2026-09-21   1986  11.45  11.48  11.35  11.48  9.541847e+08   
            2026-09-22   1987  11.45  11.50  11.37  11.46  8.877031e+08   
            2026-09-23   1988  11.43  11.46  11.30  11.35  1.052015e+09   
            2026-09-24   1989  11.35  11.47  11.29  11.30  1.186737e+09   
            2026-09-28   1990  11.28  11.41  11.27  11.30  8.105758e+08   
            2026-09-29   1991  11.30  11.41  11.28  11.35  7.846328e+08   
            2026-09-30   1992  11.36  11.65  11.33  11.57  1.205815e+09   
            2026-10-08   1993  11.57  11.87  11.54  11.78  1.743403e+09   
            2026-10-09   1994  11.78  11.89  11.58  11.59  1.259717e+09   
            2026-10-12   1995  11.62  11.83  11.62  11.73  1.243460e+09   
            2026-10-13   1996  11.73  11.77  11.55  11.57  1.157375e+09   
            2026-10-14   1997  11.57  11.72  11.48  11.50  1.122284e+09   
            2026-10-15   1998  11.50  11.52  11.26  11.41  1.719701e+09   
            2026-10-16   1999  11.41  11.63  11.36  11.50  1.281599e+09   

                             volume        sar  change_percent_1  \
code        date                                                   
000001.XSHE 2026-09-11   85079698.0  11.787395          0.870322   
            2026-09-14   75757507.0  11.779647         -0.172563   
            2026-09-15   77188695.0  11.772054         -1.037165   
            2026-09-16   97054320.0  11.764613         -0.786026   
            2026-09-17   70716600.0  11.746828          0.792254   
            2026-09-18   87182717.0  11.729755          0.262009   
            2026-09-21   83556314.0  11.704570         -0.174216   
            2026-09-22   77618545.0  11.680896         -0.959860   
            2026-09-23   92669758.0  11.658642         -0.440529   
            2026-09-24  104381872.0  11.629951          0.000000   
            2026-09-28   71534143.0  11.595955          0.442478   
            2026-09-29   69097909.0  11.556841          1.938326   
            2026-09-30  104535745.0  11.270000          1.815039   
            2026-10-08  148168594.0  11.277600         -1.612903   
            2026-10-09  107810561.0  11.301296          1.207938   
            2026-10-12  105863456.0  11.336618         -1.364024   
            2026-10-13   99493686.0  11.369821         -0.605013   
            2026-10-14   96908192.0  11.401032         -0.782609   
            2026-10-15  151176706.0  11.890000          0.788782   
            2026-10-16  111185079.0  11.877400               NaN   

                        change_percent_2  change_percent_3  change_percent_4  \
code        date                                                               
000001.XSHE 2026-09-11          0.696258         -0.348129         -1.131419   
            2026-09-14         -1.207938         -1.984469         -1.207938   
            2026-09-15         -1.815039         -1.037165         -0.777874   
            2026-09-16          0.000000          0.262009          0.087336   
            2026-09-17          1.056338          0.880282         -0.088028   
            2026-09-18          0.087336         -0.873362         -1.310044   
            2026-09-21         -1.132404         -1.567944         -1.567944   
            2026-09-22         -1.396161         -1.396161         -0.959860   
            2026-09-23         -0.440529          0.000000          1.938326

**没有规律**